# Lesson 01: Exploratory Data Analysis

**Dataset:** Amazon Reviews 2023, CDs & Vinyl: about 4.8 M ratings from 1.75 M users on 700 k albums, 1997–2023.
**Goal:** understand the data well enough to make three decisions in lesson 02: *what to predict*, *which users and albums to keep*, and *how to split train/test*.

EDA follows a standard order, and this notebook follows it too:

| # | Stage | Question it answers |
|---|---|---|
| 1 | **Data quality** | Can I trust these columns? What's missing, duplicated or malformed? |
| 2 | **Univariate distributions** | What does each variable look like on its own: centre, spread, skew, tails? |
| 3 | **Time** | Is the data stable over time, or does it drift? |
| 4 | **Relationships** | How do variables move together? (correlations, group comparisons) |
| 5 | **Dimensionality reduction (PCA)** | How many independent "directions" of variation are really there? |
| 6 | **Findings** | What does all this mean for the model? |

**How to work through it**
- Read the explanation, **predict** the result, then write your code in the `# YOUR CODE` cell.
- Run the check cell under it. It prints ✓ or says the value *looks off*, and never reveals the answer. Charts have no checks; judge them yourself using the "how to read it" notes.
- Hints are folded under **Hint**. Try without them first.
- `checks_01.py` computes the answers. Don't open it until you've finished.

In [ ]:
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import seaborn as sns
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.preprocessing import StandardScaler

import checks_01 as check
import viz  # chart styling only: viz.style(), viz.new_ax(), viz.label(...), viz.heatmap(...), viz.CATEGORICAL

viz.style()
DATA = check.DATA  # data/processed/ inside the repo
print(DATA)

## 0. Data dictionary

Start any EDA by writing down what each column *should* be, so you can spot when it isn't.

**`ratings.parquet`**: one row per review (4.8 M rows)

| Column | Type | Meaning |
|---|---|---|
| `user_id` | string | Anonymous Amazon reviewer id |
| `item_id` | string | Album id (Amazon `parent_asin`, which groups a product's editions) |
| `rating` | float | Stars, 1–5 (whole numbers only; the download script verified this) |
| `ts` | datetime | When the review was posted |
| `verified_purchase` | bool | Amazon confirmed the reviewer bought it |
| `helpful_vote` | int | How many readers marked the review helpful |
| `text_len` | int | Length of the review text in characters (the text itself is in `reviews.parquet`, which isn't needed here) |

**`albums.parquet`**: one row per album (700 k rows)

| Column | Type | Meaning |
|---|---|---|
| `item_id` | string | Joins to `ratings.item_id` |
| `title` | string | Album title |
| `store` | string | Artist, plus noise such as `"Format: Audio CD"` |
| `categories` | list[str] | Genre path, e.g. `['CDs & Vinyl', 'Jazz', 'Swing Jazz']` |
| `details` | JSON text | A bag of optional fields: label, number of discs, release dates… |
| `average_rating`, `rating_number` | float, int | Amazon's own aggregates (they include ratings outside this dataset) |
| `image_url` | string | Cover image |

---
## 1. Data quality

Before looking for patterns, check that the data means what you think it means. Three questions:
1. **Structure:** are the types right, and how much memory does it take?
2. **Completeness:** what's missing, and *why*?
3. **Validity:** duplicates, impossible values, messy text.

#### Exercise 1: Load the data

Load `ratings.parquet` into `ratings` and `albums.parquet` into `albums` (both live in `DATA`).
Print the shape of each and `.head()`.

<details><summary>Hint</summary>

`pd.read_parquet(DATA / 'ratings.parquet')`

</details>

In [ ]:
# YOUR CODE

In [ ]:
check.ex1_loaded(ratings, albums)

### 1.1 Types and memory

A DataFrame's memory use is mostly its **string columns**: each Python string is a separate object. On an 8 GB machine that matters.
Two standard fixes:
- **`category` dtype.** Store each distinct string once and keep small integer codes per row. Ideal for columns with many repeats, such as `user_id` (1.75 M distinct values over 4.8 M rows).
- **Load only the columns you need** (`pd.read_parquet(..., columns=[...])`).

#### Exercise 2: Types and memory (no check)

1. Run `ratings.info(memory_usage="deep")`. Which column is the heaviest?
2. Convert `user_id` and `item_id` to `category` in a *copy*, and compare the memory (`.memory_usage(deep=True).sum() / 1e6` MB).
3. Write one sentence: when would you *not* want `category`?

In [ ]:
# YOUR CODE

### 1.2 Missing values: how much, and *why*

Missingness has three textbook types (Rubin, 1976), and the type decides what you're allowed to do about it:

| Type | Meaning | Example here | Safe to drop or impute? |
|---|---|---|---|
| **MCAR**: missing completely at random | Unrelated to anything | A scraper glitch drops random cover URLs | Yes, dropping only loses data |
| **MAR**: missing at random | Explained by *other observed* columns | Older albums lack `Original Release Date` because Amazon added the field later | Impute using those columns |
| **MNAR**: missing not at random | Depends on the missing value *itself* | … | Dangerous: ignoring it biases results |

⭐ **The key idea for recommenders:** the ratings matrix is ~99.99% missing, and that missingness is **MNAR**. People don't rate random albums; they rate what they chose to buy, which is usually what they expected to like. So "not rated" doesn't mean "unknown opinion"; it leans towards "not interested". Keep this in mind for lesson 02.

**How to read a missing-value chart:** a bar per column showing % missing. Anything above ~5% needs an explanation; anything above ~50% is usually unusable as a feature.

#### Exercise 3: Missing values per column

Compute `missing`: a Series with one entry per column of `albums`, giving the **fraction** (0–1) of rows that are missing (`NaN`/`None`).
Then plot it as a horizontal bar chart, sorted.

Also count the albums whose `details` is the empty JSON object `'{}'`. `isna()` doesn't catch this "disguised missing" value.

<details><summary>Hint</summary>

`albums.isna().mean()`. For the chart: `missing.sort_values().plot.barh(ax=viz.new_ax())`, then `viz.label(ax, ...)`.

</details>

In [ ]:
# YOUR CODE

In [ ]:
check.ex3_missing(missing)

### 1.3 Duplicates

A **duplicate** here is the same user rating the same album more than once. It could be a re-review, a different edition grouped under one `parent_asin`, or a scraping artefact. It matters because a model would count that user's opinion twice.

#### Exercise 4: Duplicate ratings

`dup_rows` = how many rows would disappear if you kept only one row per (`user_id`, `item_id`) pair.
Then look at 10 duplicated pairs (`ratings[ratings.duplicated(["user_id", "item_id"], keep=False)]`): same rating? same day? Write down what you think they are.

In [ ]:
# YOUR CODE

In [ ]:
check.ex4_duplicates(dup_rows)

### 1.4 Nested fields: parsing `details`

Real data hides columns inside JSON blobs. `details` holds a variable set of keys per album. Pulling out the useful ones is **feature extraction**:

| New column | JSON key | Example |
|---|---|---|
| `label` | `"Label"` | `"Sony Legacy"` |
| `n_discs` | `"Number of discs"` | `"1"` |
| `original_year` | `"Original Release Date"` | `"1997"` |
| `first_available` | `"Date First Available"` | `"December 7, 2006"` |

A **missing-value matrix** (rows = a sample of albums, columns = fields, dark = missing) shows whether fields go missing *together*. Stripes spanning several columns suggest one cause (MAR); scattered dots suggest noise (MCAR).

#### Exercise 5: Parse `details` and measure its missingness

1. Build `details_df`: one row per album (same order as `albums`), with the four columns above. A field is **missing** when its key is absent **or** its value is JSON `null`.
2. `details_missing` = a Series with the fraction missing for each of the four new columns (index: `label`, `n_discs`, `original_year`, `first_available`).
3. Plot the missing-value matrix for 300 random albums: `viz.heatmap(details_df.isna().sample(300).astype(int), "…", diverging=False, fmt="")`. Hide the annotations if they clutter it: `sns.heatmap(..., cbar=False)` also works.
4. Is `original_year` MCAR, MAR or MNAR? Compare missingness across `first_available` years.

<details><summary>Hint</summary>

`json.loads(s)` turns the text into a dict; `d.get('Label')` returns `None` for an absent key. `albums.details.map(...)` applies a function row by row.

</details>

In [ ]:
# YOUR CODE

In [ ]:
check.ex5_details_missing(details_missing)

### 1.5 Messy text fields

`store` mixes the artist with format noise: `"SWV   Format: Audio CD"`, `"Shrimp City Slim  (Artist)    Format: Audio CD"`.
Cleaning text is mostly pattern spotting: look at 30 random values *before* writing the rule.

#### Exercise 6: Clean the artist name

Write `clean_artist(s: str) -> str` that returns just the artist (`"SWV"`, `"Shrimp City Slim"`). Then add `albums["artist"]` (watch out for `None`) and show the 15 artists with the most **ratings** (not albums).

<details><summary>Hint</summary>

Cut at the first `"("` and at `"Format:"`, then `.strip()`.

</details>

In [ ]:
# YOUR CODE

In [ ]:
check.ex6_clean_artist(clean_artist)

---
## 2. One variable at a time

For every variable, describe its **shape** (symmetric or skewed? one peak or several?), **centre** (mean vs median) and **spread** (std, IQR, range), and its **tails** (outliers, heavy tail?).

Rules of thumb:
- **Mean ≫ median** means right skew: a few huge values pull the mean up. Report the median.
- **Heavy-tailed counts** (reviews per user, votes, sales) look like one bar on a linear histogram. Use a **log axis**, or transform with `np.log1p(x)` = log(1 + x), which keeps 0 valid.
- **Histogram vs ECDF:** a histogram depends on your bin choice. An ECDF (empirical cumulative distribution function: for each x, the share of values ≤ x) has no bins, and you can read percentiles straight off it: `sns.ecdfplot`.

#### Exercise 7: The rating distribution

`rating_dist` = the fraction of rows per star value (a Series indexed by star, summing to 1). Bar-plot it, and print `ratings.rating.agg(["mean", "median", "std", "skew"])`.

**Predict first:** which star is most common, and roughly what share?
**How to read it:** a J-shape (a spike at 5★ and a small bump at 1★) is typical of self-selected ratings. People review what they loved or what angered them.

In [ ]:
# YOUR CODE

In [ ]:
check.ex7_rating_dist(rating_dist)

#### Exercise 8: Helpful votes and review length

1. `share_helpful` = the fraction of reviews with at least one helpful vote.
2. Plot `helpful_vote` and `text_len` as histograms of `np.log1p(...)`, side by side (`fig, axes = plt.subplots(1, 2)`).
3. Compare mean vs median for both. What does the gap tell you?

In [ ]:
# YOUR CODE

In [ ]:
check.ex8_share_helpful(share_helpful)

### 2.1 Activity: the power law

Count ratings per user and per album. Both are usually **power-law-like**: plotted on log–log axes, the histogram is roughly a straight line going down. Most users rate once; a handful rate thousands of times.

This matters because collaborative filtering learns from overlap. A user with one rating shares almost nothing with anyone.

#### Exercise 9: Ratings per user and per album

`user_counts` = number of ratings per user (a Series indexed by `user_id`); `item_counts` = the same per album.
Plot both as ECDFs with a log x-axis on one chart (`sns.ecdfplot(user_counts, label="users", ax=ax)`, then the same for albums, `ax.legend()`), and print their medians and 99th percentiles.

<details><summary>Hint</summary>

`ratings.user_id.value_counts()`

</details>

In [ ]:
# YOUR CODE

In [ ]:
check.ex9_counts(user_counts, item_counts)

### 2.2 Concentration: the Lorenz curve and the Gini coefficient

How unequal is attention across albums? Economists measure income inequality this way, and it transfers directly:
- **Lorenz curve:** sort albums from least to most rated. Plot x = cumulative share of albums against y = cumulative share of ratings. Perfect equality is the diagonal; the further the curve sags below it, the more unequal.
- **Gini coefficient:** twice the area between the diagonal and the curve. 0 means everyone equal; values near 1 mean one album gets everything.

With counts $x_{(1)} \le \dots \le x_{(n)}$ sorted ascending:
$$G = \frac{2\sum_{i=1}^{n} i\,x_{(i)}}{n\sum_{i=1}^{n} x_{(i)}} - \frac{n+1}{n}$$

**Why a recommender cares:** high Gini means a "recommend the hits" baseline will score well on accuracy while ignoring most of the catalogue. That's **popularity bias**, and in lesson 07 you'll measure whether your models make it worse.

#### Exercise 10: Lorenz curve and Gini of album popularity

Compute `gini` for `item_counts` using the formula above, and plot the Lorenz curve with the equality diagonal.
**Predict first:** a number between 0 and 1.

<details><summary>Hint</summary>

`x = np.sort(item_counts.to_numpy()).astype(float)`; `np.cumsum(x) / x.sum()` gives the y of the Lorenz curve.

</details>

In [ ]:
# YOUR CODE

In [ ]:
check.ex10_gini(gini)

---
## 3. Time

Data collected over 26 years isn't one population. Platforms change, and so does who reviews and how. This is **non-stationarity**, and it's why recommenders are evaluated with a **temporal split** (train on the past, test on the future) rather than a random one. A random split lets the model "see the future", which inflates its scores.

#### Exercise 11: Activity and rating over time

1. `per_year` = the number of ratings per calendar year (a Series indexed by int year). Plot it.
2. On a second chart, plot the **mean rating per year** and the **share of verified purchases per year**.
3. Where would you place a train/test cut-off? Why is 2023 a bad test year? (Look at how the data ends.)

In [ ]:
# YOUR CODE

In [ ]:
check.ex11_per_year(per_year)

---
## 4. Relationships between variables

### 4.1 Choosing the unit of analysis: feature tables

`ratings` has one row per *review*. Questions like "are popular albums rated higher?" are about **albums**, so first aggregate to one row per album, and likewise per user. This is **feature engineering**, and these tables will later feed models directly.

#### Exercise 12: Album feature table

Build `album_feats`: one row per **rated** album, indexed by `item_id`, with the columns:

| Column | Definition |
|---|---|
| `n_ratings` | number of rating rows (duplicates included) |
| `mean_rating` | mean stars |
| `std_rating` | *sample* standard deviation of stars (`NaN` when there's only 1 rating) |
| `pct_verified` | share of verified purchases (0–1) |
| `mean_helpful` | mean helpful votes |
| `mean_text_len` | mean review length |

<details><summary>Hint</summary>

One `groupby('item_id').agg(...)` with named aggregations: `n_ratings=('rating', 'size')`, …

</details>

In [ ]:
# YOUR CODE

In [ ]:
check.ex12_album_feats(album_feats)

#### Exercise 13: User feature table

Build `user_feats` indexed by `user_id` with `n_ratings`, `mean_rating`, `std_rating`, `pct_verified`, `mean_text_len` (same definitions as above) and
`days_active` = days between the user's first and last rating, as a float (e.g. 1.5 for a day and a half).
Then look at `user_feats.describe()`: what does a typical user look like?

<details><summary>Hint</summary>

Aggregate `first=('ts', 'min')` and `last=('ts', 'max')`, then `(last - first).dt.total_seconds() / 86400`.

</details>

In [ ]:
# YOUR CODE

In [ ]:
check.ex13_user_feats(user_feats)

### 4.2 Correlation: Pearson vs Spearman

A **correlation coefficient** summarises how two variables move together, from −1 (perfectly opposite) through 0 (no relationship) to +1 (perfectly together).

| | **Pearson r** | **Spearman ρ** |
|---|---|---|
| Measures | *linear* association | *monotonic* association (always up, or always down) |
| Computed on | raw values | ranks |
| Sensitive to outliers and skew | very | barely |
| Use it when | both variables are roughly bell-shaped | counts, heavy tails, ordinal data such as stars |

Most variables here are heavy-tailed, so **Spearman is the honest default**. A large gap between the two on the same pair means outliers or non-linearity. Look at the scatter plot.

**Two traps**
- **Small-sample noise.** An album with 1 rating has a mean of exactly 1, 2, 3, 4 or 5. Those extreme means are noise, not signal. Filter to albums with enough ratings first (the idea behind *shrinkage*, which returns later).
- **Correlation ≠ causation.** Verified buyers might write shorter reviews because *casual buyers* write short reviews, not because verification shortens text.

**How to read a correlation heatmap:** look at the **strongest** cells, not every one. With thousands of rows almost every |r| > 0.05 is "statistically significant" but practically meaningless. Treat |ρ| > 0.3 as worth a sentence and > 0.7 as close to redundant.

#### Exercise 14: Correlations between album features

1. `album_feats_10` = the rows of `album_feats` with `n_ratings >= 10`.
2. `corr_pearson` and `corr_spearman` = the two correlation matrices of `album_feats_10`. Plot both with `viz.heatmap(...)`, side by side if you like.
3. Scatter plot: popularity (`n_ratings`, log x-axis) vs `mean_rating`. With 90 k points use `ax.hexbin(np.log10(x), y, gridsize=50, cmap=viz.SEQUENTIAL, mincnt=1)`, or `alpha=0.05` dots.
4. Find the strongest correlation. Is there a mechanical explanation? (Hint: what can `std_rating` be when `mean_rating` is 5.0?)

In [ ]:
# YOUR CODE

In [ ]:
check.ex14_corr(album_feats_10, corr_spearman)

### 4.3 Numeric vs categorical: ratings by genre

Correlation needs two numbers. To compare a number across **groups**, use box plots:
- The **box** spans the 25th–75th percentile (the IQR), and the line is the median.
- The **whiskers** extend to 1.5 × IQR; dots beyond are outliers.

Sort the boxes by median, or the chart is unreadable. Keep ≤ 12 groups and fold the rest into "Other".

#### Exercise 15: Mean rating by genre (no check)

1. Add a `genre` column to `albums` = `categories[1]` (the entry after `'CDs & Vinyl'`), and count albums per genre. Which "genres" aren't genres at all? (Look for store sections.)
2. Join `genre` onto `album_feats_10` and draw `sns.boxplot(data=..., x="mean_rating", y="genre", order=...)` for the 12 biggest genres, sorted by median.

In [ ]:
# YOUR CODE

---
## 5. Dimensionality reduction: PCA

### The idea
You have *p* correlated features. **Principal Component Analysis** finds new axes, the **principal components**, such that:
1. PC1 is the direction along which the data varies the most;
2. PC2 is the direction of greatest *remaining* variance, perpendicular (uncorrelated) to PC1, and so on.

Mathematically, the components are the **eigenvectors** of the features' covariance matrix, and each eigenvalue is the variance along its component. The **explained variance ratio** of a component is its eigenvalue divided by the sum of all eigenvalues.

### What to look at
- **Scree plot.** Explained variance per component (plus the running total). An "elbow", or reaching ~80% cumulative, tells you how many independent directions the data really has.
- **Loadings.** Each component is a weighted mix of the original features. Big positive or negative weights tell you what the component *means*, so name it: "engagement", "polarisation"…
- **Projection.** Plot every album at (PC1, PC2) and colour by genre. Do genres separate?

### Two rules before running PCA
- **Standardise** (mean 0, std 1 per feature). PCA chases variance, so unscaled, `n_ratings` (range 10–10,000) would swamp `pct_verified` (0–1).
- **Tame heavy tails first** (`np.log1p`). A few extreme albums would otherwise define PC1 on their own.

#### Exercise 16: PCA on album features

1. Build a feature matrix from `album_feats_10` plus `original_year` (from `details_df`, converted with `pd.to_numeric(..., errors="coerce")` and joined on `item_id`). Drop rows with any `NaN`.
   Apply `np.log1p` to `n_ratings`, `mean_helpful` and `mean_text_len`.
2. `X_std` = the standardised matrix (`StandardScaler().fit_transform(...)`); `pca` = `PCA()` (all components) fitted on `X_std`.
3. Scree plot: bars of `pca.explained_variance_ratio_`, plus a line of the cumulative sum.
4. Loadings: `viz.heatmap(pd.DataFrame(pca.components_[:3].T, index=<feature names>, columns=["PC1", "PC2", "PC3"]), "Loadings")`. Name PC1 and PC2.
5. Projection: scatter PC1 vs PC2 for the 6 biggest genres (colour = genre, in `viz.CATEGORICAL` order; everything else grey, drawn first).

In [ ]:
# YOUR CODE

In [ ]:
check.ex16_pca(X_std, pca)

### 5.1 The same idea on the ratings matrix: your first recommender

Now apply the idea to the **user × album matrix** *R* (1 if the user rated the album, else 0). It's huge and almost empty, so instead of PCA on a covariance matrix you use **truncated SVD**, which works directly on sparse matrices:

$$R \approx U_k \, \Sigma_k \, V_k^\top$$

- Each album gets a *k*-dimensional vector (a row of $V_k$), its **embedding**. Albums rated by the same people end up close together.
- **Cosine similarity** between two embeddings measures how similar their audiences are, not their sound. Sorting by it gives "people who liked X also liked…": an item-to-item recommender.
- This is exactly what **matrix factorization** (lesson 05) optimises properly. Here you're just looking at it.

Expect the explained variance to be *tiny* (a few %). Taste is high-dimensional, and a 0/1 matrix is mostly noise. That doesn't make the embeddings useless. Judge them by their neighbours.

#### Exercise 17: Build the interaction matrix and find neighbours

1. Keep only rating rows whose **user has ≥ 5 ratings and whose album has ≥ 20 ratings**, both counted on the **full** `ratings` table (one pass, not repeated). Then keep each (user, album) pair once.
2. `R` = a `scipy.sparse.csr_matrix` of shape (number of kept users, number of kept albums), with 1 for each kept pair.
3. `svd = TruncatedSVD(n_components=20, random_state=0).fit(R)`; album embeddings = `svd.components_.T`. Normalise each row to length 1.
4. Write `neighbours(title, k=8)`: find the album by title, return the *k* albums with the highest cosine similarity. Try `"Abbey Road"`, `"Kind of Blue"` and one album you know well. Do the neighbours make sense? What kind of mistakes does it make?

<details><summary>Hint</summary>

Map ids to row and column numbers with `.astype('category').cat.codes`; then `sp.csr_matrix((np.ones(n), (rows, cols)), shape=(n_users, n_items))`.

</details>

In [ ]:
# YOUR CODE

In [ ]:
check.ex17_matrix(R)

---
## 6. Findings

Fill in this table. It's the summary of the whole EDA and the input to lesson 02.

| Finding | Number | What it means for the recommender |
|---|---|---|
| Sparsity (density of *R*) | | |
| Rating skew (share of 5★) | | |
| Median ratings per user / per album | | |
| Gini of album popularity | | |
| Duplicate rows | | |
| Strongest album-feature correlation | | |
| Components needed for 80% variance (album PCA) | | |
| Best train/test cut-off year | | |

## Reflect
*Answer in this cell. These answers become README and interview material.*

1. Given the rating distribution and the MNAR point in §1.2, would you rather predict the *star value* or *whether a user rates an album at all*? Why?
2. What would "recommend the top 10 most popular albums to everyone" get right, and what would it get wrong? Use your Gini number.
3. Your SVD neighbours in exercise 17: name one good and one bad neighbour, and explain *why* the model made each.